# Macroeconomic Factors and Demographic Outcomes in Europe

This notebook analyzes how macroeconomic indicators (GDP, inflation, housing prices, unemployment) affect birth and marriage rates.

## 1. Import Libraries

In [1]:
import pandas as pd
import statsmodels.formula.api as smf
import numpy as np
from sklearn.preprocessing import StandardScaler

## 2. Load Datasets

In [2]:
birth = pd.read_csv("DATA_RAW/birth.csv")
marriage = pd.read_csv("DATA_RAW/marriage.csv")
gdp = pd.read_csv("DATA_RAW/gdp.csv")
hicp = pd.read_csv("DATA_RAW/hicp.csv")
hpi = pd.read_csv("DATA_RAW/hpi.csv")
unemp = pd.read_csv("DATA_RAW/unemp_cit.csv")

## 3. Clean and Prepare Data

In [3]:
def clean(df, value_name):
    return df[["geo", "TIME_PERIOD", "OBS_VALUE"]].rename(columns={
        "geo": "country",
        "TIME_PERIOD": "year",
        "OBS_VALUE": value_name
    })

birth_df = clean(birth, "birth_rate")
marriage_df = clean(marriage, "marriage_rate")
gdp_df = clean(gdp, "gdp")
hicp_df = clean(hicp, "hicp_index")
hpi_df = clean(hpi, "house_price_index")
unemp_df = clean(unemp, "unemployment_rate")

for df in [birth_df, marriage_df, gdp_df, hicp_df, hpi_df, unemp_df]:
    df["year"] = pd.to_numeric(df["year"], errors="coerce")
    df.iloc[:, 2] = pd.to_numeric(df.iloc[:, 2], errors="coerce")

## 4. Compute Inflation

In [4]:
hicp_df = hicp_df.sort_values(["country", "year"])
hicp_df["inflation_rate"] = hicp_df.groupby("country")["hicp_index"].pct_change(fill_method=None)

## 5. Merge Datasets

In [5]:
df = birth_df.merge(marriage_df, on=["country", "year"]) \
    .merge(gdp_df, on=["country", "year"]) \
    .merge(hicp_df, on=["country", "year"]) \
    .merge(hpi_df, on=["country", "year"]) \
    .merge(unemp_df, on=["country", "year"])

df = df.dropna()

## 6. Feature Engineering

In [6]:
df = df.sort_values(["country", "year"])
df["lag_house_price"] = df.groupby("country")["house_price_index"].shift(1)
df["lag_inflation"] = df.groupby("country")["inflation_rate"].shift(1)

df = df.replace([np.inf, -np.inf], np.nan)

df["log_gdp"] = np.log(df["gdp"].clip(lower=1))
df["log_house_price"] = np.log(df["house_price_index"].clip(lower=1))

model_vars = [
    "birth_rate", "marriage_rate",
    "lag_house_price", "lag_inflation",
    "gdp", "unemployment_rate"
]

df = df.dropna(subset=model_vars)

## 7. Normalize Variables

In [7]:
cols = ["log_house_price", "lag_inflation", "log_gdp", "unemployment_rate"]

scaler = StandardScaler()
df[cols] = scaler.fit_transform(df[cols])

df["lag_inflation"] = df["lag_inflation"].clip(-1, 1)

## 8. Birth Rate Regression

In [8]:
model_birth = smf.ols(
    formula="birth_rate ~ lag_house_price + lag_inflation + gdp + unemployment_rate",
    data=df
).fit()

print(model_birth.summary())

                            OLS Regression Results                            
Dep. Variable:             birth_rate   R-squared:                       0.020
Model:                            OLS   Adj. R-squared:                  0.020
Method:                 Least Squares   F-statistic:                     6968.
Date:                Sun, 26 Apr 2026   Prob (F-statistic):               0.00
Time:                        10:49:23   Log-Likelihood:            -1.8879e+07
No. Observations:             1337476   AIC:                         3.776e+07
Df Residuals:                 1337471   BIC:                         3.776e+07
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept          9.695e+04    387.42

## 9. Marriage Rate Regression

In [9]:
model_marriage = smf.ols(
    formula="marriage_rate ~ lag_house_price + lag_inflation + gdp + unemployment_rate",
    data=df
).fit()

print(model_marriage.summary())

                            OLS Regression Results                            
Dep. Variable:          marriage_rate   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                  0.001
Method:                 Least Squares   F-statistic:                     181.6
Date:                Sun, 26 Apr 2026   Prob (F-statistic):          6.97e-156
Time:                        10:49:24   Log-Likelihood:            -6.3733e+06
No. Observations:             1337476   AIC:                         1.275e+07
Df Residuals:                 1337471   BIC:                         1.275e+07
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept            28.9520      0.03

## 10. Fixed Effects Model

In [10]:
model_fe = smf.ols(
    formula="birth_rate ~ log_house_price + lag_inflation + log_gdp + unemployment_rate + C(country) + C(year)",
    data=df
).fit(cov_type='cluster', cov_kwds={'groups': df['country']})

print(model_fe.summary())

                            OLS Regression Results                            
Dep. Variable:             birth_rate   R-squared:                       0.444
Model:                            OLS   Adj. R-squared:                  0.444
Method:                 Least Squares   F-statistic:                 3.083e+04
Date:                Sun, 26 Apr 2026   Prob (F-statistic):           3.76e-60
Time:                        10:49:43   Log-Likelihood:            -1.8501e+07
No. Observations:             1337476   AIC:                         3.700e+07
Df Residuals:                 1337430   BIC:                         3.700e+07
Df Model:                          45                                         
Covariance Type:              cluster                                         
                                                              coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------

c:\Users\hamin\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 45, but rank is 14
  warnings.warn('covariance of constraints does not have full '
